# 02 Exploratory Data Analysis

This notebook performs the EDA stage for Project 01: EIA-861M Electricity Retail Sales Analytics.

The EDA is guided by `../docs/eda_design.md`. The goal is to build decision-relevant understanding of state-sector electricity sales, revenue, customer base, and average retail price patterns for the Project 01 dashboard and investigation brief. Forecasting is reserved for Project 02.

## Scope

This notebook starts from the flagged interim dataset produced by the acquisition-validation workflow:

```text
data/interim/eia861m_retail_sales_2016_2025_with_quality_flags.csv
```

It does not call the EIA API. If the interim dataset is missing, run `notebooks/01_data_acquisition_validation.ipynb` or `scripts/download_data.py` followed by `scripts/validate_data.py` first.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# Colab convenience: mount Google Drive when this notebook runs in Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

RECOMMENDED_COLAB_ROOT = Path(
    "/content/drive/MyDrive/ds_portfolio/project_01_eia861m_electricity_sales"
)

PROJECT_ROOT_OVERRIDE = RECOMMENDED_COLAB_ROOT if RECOMMENDED_COLAB_ROOT.exists() else None


def find_project_root() -> Path:
    """Locate the project root containing src/eia861m and configs/config.json."""
    candidates = []

    env_root = os.getenv("EIA861M_PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())

    if PROJECT_ROOT_OVERRIDE is not None:
        candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    for candidate in candidates:
        if (candidate / "src" / "eia861m").exists() and (candidate / "configs" / "config.json").exists():
            return candidate.resolve()

    searched = "\n".join(str(candidate) for candidate in candidates)
    raise RuntimeError(
        "Could not locate the project root. Upload/sync the full project folder, not only this notebook.\n\n"
        f"Current working directory: {cwd}\n\n"
        f"Searched candidate paths:\n{searched}"
    )


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from eia861m.config import load_config
from eia861m.eda import (
    add_safe_ratio_features,
    add_time_features,
    build_national_annual,
    build_national_monthly,
    build_sector_monthly,
    build_state_annual,
    build_state_sector_annual,
)
from eia861m.paths import project_path

print(f"Project root: {PROJECT_ROOT}")

## Load Flagged Interim Dataset

The flagged interim dataset is the source of truth for EDA. It preserves published EIA values and carries data-quality flags from the validation stage.

In [ ]:
config = load_config()
interim_path = project_path(config["paths"]["validation_flags"])

if not interim_path.exists():
    raise FileNotFoundError(
        f"Flagged interim dataset not found: {interim_path}. "
        "Run the acquisition-validation workflow first."
    )

df = pd.read_csv(interim_path)
print(f"Loaded: {interim_path}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

In [ ]:
df.head()

## Input Sanity Checks

These checks confirm that the EDA is using the expected validated dataset, not an accidental raw or manually edited file.

In [ ]:
expected_rows = config["validation"]["expected_rows"]
required_flags = [
    "flag_all_metrics_zero",
    "flag_any_metric_zero",
    "flag_negative_revenue",
    "flag_transportation_sector",
    "flag_revenue_consistency_check",
]

missing_flags = [column for column in required_flags if column not in df.columns]

assert len(df) == expected_rows, f"Unexpected row count: {len(df)} != {expected_rows}"
assert not missing_flags, f"Missing quality flag columns: {missing_flags}"
assert df.duplicated(subset=["period", "stateid", "sectorid"]).sum() == 0

pd.Series(
    {
        "rows": len(df),
        "unique_months": df["period"].nunique(),
        "unique_states": df["stateid"].nunique(),
        "unique_sectors": df["sectorid"].nunique(),
        "duplicate_keys": df.duplicated(subset=["period", "stateid", "sectorid"]).sum(),
    }
)

In [ ]:
df[required_flags].sum().astype(int)

## Prepare EDA Fields

Time and ratio features are added for analysis. Ratio metrics are created carefully because `customers` or `sales` can be zero, especially in `TRA`.

In [ ]:
df_eda = add_time_features(df)
df_eda = add_safe_ratio_features(df_eda)

df_eda[["period", "period_month", "year", "month", "month_name"]].head()

## National Monthly Overview

Start at the broadest level before drilling into sector, state, and state-sector patterns.

In [ ]:
national_monthly = build_national_monthly(df_eda)
national_monthly.head()

In [ ]:
national_monthly.describe(include="all")

## Sector Monthly Overview

Sector-level trends help separate broad electricity market patterns from sector-specific behavior. The 2016-2025 sector summary sums sales and revenue but reports the average monthly customer count, not a sum of customer counts across months.

In [ ]:
sector_monthly = build_sector_monthly(df_eda)
sector_monthly.head(12)

In [ ]:
sector_totals = (
    sector_monthly.groupby(["sectorid", "sectorName"], as_index=False)
    .agg(
        sales=("sales", "sum"),
        revenue=("revenue", "sum"),
        avg_monthly_customers=("customers", "mean"),
    )
)
sector_flags = (
    df_eda.groupby(["sectorid", "sectorName"], as_index=False)
    .agg(
        zero_flag_count=("flag_any_metric_zero", "sum"),
        negative_revenue_count=("flag_negative_revenue", "sum"),
    )
)
sector_totals = sector_totals.merge(sector_flags, on=["sectorid", "sectorName"])
sector_totals["sales_share"] = sector_totals["sales"] / sector_totals["sales"].sum()
sector_totals["revenue_share"] = sector_totals["revenue"] / sector_totals["revenue"].sum()
sector_totals["implied_price_cents_kwh"] = 100 * sector_totals["revenue"] / sector_totals["sales"]
sector_totals.sort_values("sales", ascending=False)

## Annual Market Summary

Annual sales and revenue are sums of monthly flows. `avg_monthly_customers` is the mean of 12 monthly customer counts; it is not a count of unique people or accounts over the year. The aggregate price is `100 * revenue / sales` in cents/kWh.

In [ ]:
annual_summary = build_national_annual(national_monthly)
annual_summary["sales_yoy_growth"] = annual_summary["sales"].pct_change()
annual_summary["revenue_yoy_growth"] = annual_summary["revenue"].pct_change()
annual_summary

## State Contribution

State-level contribution identifies high-impact geographies. State-year sales and revenue are summed; customer counts are averaged across the 12 months. This is descriptive prioritization, not causal explanation.

In [ ]:
state_annual = build_state_annual(df_eda)
latest_year = int(state_annual["year"].max())

state_latest = state_annual[state_annual["year"] == latest_year].copy()
state_latest["sales_share"] = state_latest["sales"] / state_latest["sales"].sum()
state_latest["revenue_share"] = state_latest["revenue"] / state_latest["revenue"].sum()
state_latest.head(15)

## State-Sector Segmentation

State-sector combinations are the natural bridge between broad market monitoring and later dashboard segmentation. `avg_monthly_customers` represents the mean monthly count in each state-sector-year.

In [ ]:
state_sector_annual = build_state_sector_annual(df_eda)
state_sector_latest = state_sector_annual[state_sector_annual["year"] == latest_year].copy()
state_sector_latest.head(20)

## Seasonality And Volatility Candidates

Calendar-month profiles and full-period variability are exploratory screens. A recurring seasonal pattern still needs confirmation across individual years. Aggregate price is calculated from total revenue and sales, not from an unweighted average of state prices.

In [ ]:
monthly_sector_profile = (
    sector_monthly.assign(month=sector_monthly["period_month"].dt.month)
    .groupby(["sectorid", "month"], as_index=False)
    .agg(
        avg_monthly_sales=("sales", "mean"),
        avg_monthly_revenue=("revenue", "mean"),
        total_sales=("sales", "sum"),
        total_revenue=("revenue", "sum"),
    )
    .sort_values(["sectorid", "month"])
)
monthly_sector_profile["implied_price_cents_kwh"] = (
    100 * monthly_sector_profile["total_revenue"] / monthly_sector_profile["total_sales"].where(monthly_sector_profile["total_sales"] != 0)
)
monthly_sector_profile = monthly_sector_profile.drop(columns=["total_sales", "total_revenue"])
monthly_sector_profile.head(16)

In [ ]:
state_sector_volatility = (
    df_eda.groupby(["stateid", "stateDescription", "sectorid", "sectorName"], as_index=False)
    .agg(
        mean_sales=("sales", "mean"),
        std_sales=("sales", "std"),
        observed_months=("period_month", "nunique"),
        zero_flag_count=("flag_any_metric_zero", "sum"),
    )
)
state_sector_volatility["sales_cv"] = (
    state_sector_volatility["std_sales"] / state_sector_volatility["mean_sales"]
)
state_sector_volatility.sort_values("sales_cv", ascending=False).head(20)

## Quality Flags In EDA

Quality flags must remain visible in interpretation, especially for `TRA` and zero-heavy observations.

In [ ]:
quality_by_sector = (
    df_eda.groupby(["sectorid", "sectorName"], as_index=False)[required_flags]
    .sum()
    .sort_values("flag_any_metric_zero", ascending=False)
)
quality_by_sector

## Candidate Forecasting Segments

This is not forecasting yet. This section only identifies possible candidates that may deserve a baseline forecasting experiment later.

In [ ]:
forecast_candidate_screen = state_sector_volatility.copy()
forecast_candidate_screen = forecast_candidate_screen[
    (forecast_candidate_screen["mean_sales"] > 0)
    & (forecast_candidate_screen["observed_months"] == df_eda["period_month"].nunique())
    & (forecast_candidate_screen["zero_flag_count"] == 0)
].copy()
forecast_candidate_screen.sort_values(["mean_sales", "sales_cv"], ascending=[False, True]).head(20)

## Selected Figures

These figures support the main descriptive questions. They are saved under reports/figures for review. Sales and revenue are annual flows; customer counts are excluded from these plots to avoid mixing flow and stock measures.

In [ ]:
import matplotlib.pyplot as plt

figures_dir = project_path("reports/figures")
figures_dir.mkdir(parents=True, exist_ok=True)
print(f"Saving EDA figures to: {figures_dir}")

### National annual trend

Separate panels show sales, revenue, and implied aggregate price without a dual axis. Revenue and implied price are not adjusted for inflation. Each panel has its own vertical scale beginning above zero; use the annotated values and percentage changes when comparing trends. The price series is a ratio of reported revenue to sales, not a tariff.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True, constrained_layout=True)
annual_series = [
    ("Sales (TWh)", annual_summary["sales"] / 1000, "#0E6C71"),
    ("Revenue (USD bn, nominal)", annual_summary["revenue"] / 1000, "#505C63"),
    ("Implied price (nominal cents/kWh)", annual_summary["implied_price_cents_kwh"], "#C65C2E"),
]
for axis, (label, values, color) in zip(axes, annual_series):
    axis.plot(annual_summary["year"], values, marker="o", linewidth=2, color=color)
    axis.set_ylabel(label)
    axis.grid(axis="y", alpha=0.2)
    change_pct = 100 * (values.iloc[-1] / values.iloc[0] - 1)
    axis.text(0.02, 0.94, f"{values.iloc[0]:,.2f} to {values.iloc[-1]:,.2f} ({change_pct:+.1f}%)", transform=axis.transAxes, va="top", fontsize=9, color=color)
axes[-1].set_xticks(annual_summary["year"])
axes[-1].set_xlabel("Year")
fig.suptitle(f"U.S. electricity retail sales | {annual_summary['year'].min()}-{annual_summary['year'].max()}")
fig.savefig(figures_dir / "01_national_annual_trends.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

### Sector mix in the latest year

Both shares use 2025. RES contributed 37.3% of sales and 47.4% of revenue; IND contributed 25.7% of sales and 16.2% of revenue. These are shares of the published EIA totals, not company market shares. The earlier sector_totals table pools 2016-2025.

In [ ]:
sector_latest_summary = (
    sector_monthly[sector_monthly["period_month"].dt.year == latest_year]
    .groupby(["sectorid", "sectorName"], as_index=False)
    .agg(sales=("sales", "sum"), revenue=("revenue", "sum"))
    .sort_values("sales", ascending=False)
)
sector_latest_summary["sales_share_pct"] = 100 * sector_latest_summary["sales"] / sector_latest_summary["sales"].sum()
sector_latest_summary["revenue_share_pct"] = 100 * sector_latest_summary["revenue"] / sector_latest_summary["revenue"].sum()
sector_latest_summary

In [ ]:
positions = list(range(len(sector_latest_summary)))
fig, axis = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
sales_bars = axis.barh([p - 0.2 for p in positions], sector_latest_summary["sales_share_pct"], height=0.36, color="#0E6C71", label="Sales")
revenue_bars = axis.barh([p + 0.2 for p in positions], sector_latest_summary["revenue_share_pct"], height=0.36, color="#C65C2E", label="Revenue")
axis.set_yticks(positions)
axis.set_yticklabels(sector_latest_summary["sectorid"])
axis.invert_yaxis()
axis.set_xlim(0, max(sector_latest_summary[["sales_share_pct", "revenue_share_pct"]].max()) * 1.2)
axis.set_xlabel("Share of national total (%)")
axis.set_title(f"Electricity retail sales and revenue by sector | {latest_year}")
axis.bar_label(sales_bars, fmt="%.1f%%", padding=3, fontsize=9)
axis.bar_label(revenue_bars, fmt="%.1f%%", padding=3, fontsize=9)
axis.legend(frameon=False)
axis.grid(axis="x", alpha=0.2)
fig.savefig(figures_dir / "02_sector_mix_latest_year.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

### State contribution in the latest year

Texas accounted for 12.8% of national sales in 2025, followed by Florida at 6.4% and California at 5.9%. This ranking indicates scale in the published EIA data, not a causal driver or investment recommendation.

In [ ]:
top_states = state_latest.nlargest(10, "sales_share").sort_values("sales_share")
fig, axis = plt.subplots(figsize=(9, 6), constrained_layout=True)
bars = axis.barh(top_states["stateDescription"], 100 * top_states["sales_share"], color="#0E6C71")
axis.set_xlim(0, 100 * top_states["sales_share"].max() * 1.2)
axis.set_xlabel("Share of national sales (%)")
axis.set_title(f"Top 10 states by electricity retail sales | {latest_year}")
axis.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
axis.grid(axis="x", alpha=0.2)
fig.savefig(figures_dir / "03_top_state_sales_share_latest_year.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

### Recurring monthly sales patterns

An index of 1.0 equals that sector's average monthly sales within the same year. Gray lines show individual years and the colored line their mean. RES reached about 1.31 in July and 1.28 in August, versus 0.77 in April; COM and IND had smaller seasonal swings. The pattern is descriptive, not evidence that a forecast will outperform a baseline. TRA is omitted because of its many zero-valued observations.

In [ ]:
seasonal_monthly = sector_monthly[sector_monthly["sectorid"].isin(["RES", "COM", "IND"])].copy()
seasonal_monthly["year"] = seasonal_monthly["period_month"].dt.year
seasonal_monthly["month"] = seasonal_monthly["period_month"].dt.month
assert seasonal_monthly.groupby(["sectorid", "year"])["month"].nunique().eq(12).all()
yearly_monthly_mean = seasonal_monthly.groupby(["sectorid", "year"])["sales"].transform("mean")
seasonal_monthly["sales_index"] = seasonal_monthly["sales"] / yearly_monthly_mean.where(yearly_monthly_mean != 0)
seasonal_profile = (
    seasonal_monthly.groupby(["sectorid", "month"], as_index=False)
    .agg(mean_sales_index=("sales_index", "mean"))
)
seasonal_profile.pivot(index="sectorid", columns="month", values="mean_sales_index").round(2)

In [ ]:
sector_colors = {"RES": "#0E6C71", "COM": "#C65C2E", "IND": "#505C63"}
fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True, sharey=True, constrained_layout=True)
for axis, sector in zip(axes, sector_colors):
    history = seasonal_monthly[seasonal_monthly["sectorid"] == sector]
    for _, year_data in history.groupby("year"):
        axis.plot(year_data["month"], year_data["sales_index"], color="#B7C2C4", alpha=0.55, linewidth=1)
    average = seasonal_profile[seasonal_profile["sectorid"] == sector]
    axis.plot(average["month"], average["mean_sales_index"], color=sector_colors[sector], linewidth=2.5, marker="o", label="Mean across years")
    axis.axhline(1, color="#505C63", linestyle="--", linewidth=0.8)
    axis.set_title(sector, loc="left")
    axis.set_ylabel("Sales index")
    axis.grid(axis="y", alpha=0.2)
axes[0].legend(frameon=False, loc="upper right")
axes[-1].set_xticks(range(1, 13))
axes[-1].set_xlabel("Calendar month")
fig.suptitle("Monthly sales relative to each sector-year average")
fig.savefig(figures_dir / "04_sector_monthly_sales_index.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## EDA Findings And Boundaries

The executed notebook, KPI audit, and reviewed figures support these descriptive observations for the current dataset snapshot:

- From 2016 to 2025, annual sales increased 7.9% and nominal revenue increased 43.1%. Implied average price rose from 10.27 to 13.63 nominal cents/kWh. Revenue and price are not inflation-adjusted; these changes do not identify a causal price effect.
- From 2024 to 2025, sales grew 2.1% and nominal revenue grew 7.6%; implied average price moved from 12.94 to 13.63 nominal cents/kWh.
- In 2025, RES accounted for 37.3% of national sales and 47.4% of revenue, while IND accounted for 25.7% of sales and 16.2% of revenue. These are 2025 shares, distinct from the pooled 2016-2025 sector summary above.
- Texas accounted for 12.8% of national sales in 2025. Texas IND, COM, and RES were the three largest state-sector combinations by annual sales.
- The 2025 average monthly customer count was about 164.5 million across the reported state-sector observations. The sum of 12 monthly counts must not be presented as unique annual customers.
- RES showed a recurring summer sales peak in the within-year normalized series, strongest in July and August. COM and IND showed smaller swings. This does not establish forecasting value.
- All 2,646 rows with at least one zero metric were in `TRA`; one `TRA` row had a tiny negative revenue. Published values remain retained and flagged.

Use these findings for market monitoring and segment prioritization. The public dataset does not establish utility-specific business impact, policy effects, or causality. Any forecasting claim requires a chronological baseline evaluation.

In [ ]:
print("EDA calculations and selected figures completed. Review the saved figures and findings before reporting results.")